In [1]:
# ============================================================
# BLOCK 1 — Install & imports
# ============================================================
!pip install -q monai nibabel

import os, glob, random, tarfile, time, shutil
import numpy as np
import torch
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    Spacingd, NormalizeIntensityd, CropForegroundd,
    Resized, ConcatItemsd, MapTransform, ToTensord
)

random.seed(42)


# ============================================================
# BLOCK 2 — Config
# ============================================================
TARGET_SIZE = (96, 96, 96)
CHUNK_ID = 1                  # <<< SET TO 0, THEN 1 >>>
N_CHUNKS = 2

BRATS_TAR = "/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar"
BRATS_EXTRACT_DIR = "/kaggle/working/brats2021_extracted"
IXI_DIR = "/kaggle/input/datasets/thintrnvn/ixi-dataset-t2-mri"

KEYS = ["t1", "t1ce", "t2", "flair", "label"]
IMG_KEYS = ["t1", "t1ce", "t2", "flair"]


# ============================================================
# BLOCK 3 — Build full patient list, slice out this chunk
# ============================================================
with tarfile.open(BRATS_TAR) as tar:
    all_members = tar.getmembers()
    all_names = [m.name for m in all_members]

def clean_path(name):
    return name.lstrip("./")

all_brats_prefixes = sorted(set(
    clean_path(n).split("/")[0] for n in all_names
    if clean_path(n).startswith("BraTS2021_")
))
print(f"Total BraTS patients available: {len(all_brats_prefixes)}")

random.Random(42).shuffle(all_brats_prefixes)
chunk_size_brats = len(all_brats_prefixes) // N_CHUNKS
start_idx = CHUNK_ID * chunk_size_brats
end_idx = len(all_brats_prefixes) if CHUNK_ID == N_CHUNKS - 1 else (CHUNK_ID + 1) * chunk_size_brats
this_chunk_prefixes = set(all_brats_prefixes[start_idx:end_idx])
print(f"Chunk {CHUNK_ID}: {len(this_chunk_prefixes)} BraTS patients (indices {start_idx}:{end_idx})")

all_ixi_folders = sorted(glob.glob(os.path.join(IXI_DIR, "IXI*-T2.nii")))
random.Random(42).shuffle(all_ixi_folders)
print(f"Total IXI subjects available: {len(all_ixi_folders)}")

chunk_size_ixi = len(all_ixi_folders) // N_CHUNKS
start_ixi = CHUNK_ID * chunk_size_ixi
end_ixi = len(all_ixi_folders) if CHUNK_ID == N_CHUNKS - 1 else (CHUNK_ID + 1) * chunk_size_ixi
this_chunk_ixi_folders = all_ixi_folders[start_ixi:end_ixi]
print(f"Chunk {CHUNK_ID}: {len(this_chunk_ixi_folders)} IXI subjects")


# ============================================================
# BLOCK 4 — Extract only this chunk's BraTS patients
# ============================================================
shutil.rmtree(BRATS_EXTRACT_DIR, ignore_errors=True)
os.makedirs(BRATS_EXTRACT_DIR, exist_ok=True)

print(f"Extracting chunk {CHUNK_ID} patients from tar...")
start_extract = time.time()
with tarfile.open(BRATS_TAR) as tar:
    members_to_extract = [m for m in all_members if clean_path(m.name).split("/")[0] in this_chunk_prefixes]
    print(f"Extracting {len(members_to_extract)} files for {len(this_chunk_prefixes)} patients...")
    tar.extractall(path=BRATS_EXTRACT_DIR, members=members_to_extract)
print(f"Extraction done in {(time.time()-start_extract)/60:.1f} min.")

def find_brats_root(base_dir):
    top_items = os.listdir(base_dir)
    patient_like = [d for d in top_items if d.startswith("BraTS2021_") and os.path.isdir(os.path.join(base_dir, d))]
    if patient_like:
        return base_dir
    for item in top_items:
        sub_path = os.path.join(base_dir, item)
        if os.path.isdir(sub_path):
            sub_items = os.listdir(sub_path)
            patient_like = [d for d in sub_items if d.startswith("BraTS2021_") and os.path.isdir(os.path.join(sub_path, d))]
            if patient_like:
                return sub_path
    raise RuntimeError("Could not locate BraTS patient folders.")

BRATS_DIR = find_brats_root(BRATS_EXTRACT_DIR)
print("Resolved BRATS_DIR:", BRATS_DIR)
!df -h /kaggle/working


# ============================================================
# BLOCK 5 — Build datalists for this chunk
# ============================================================
def build_brats_datalist(brats_dir):
    patient_dirs = sorted(glob.glob(os.path.join(brats_dir, "BraTS2021_*")))
    datalist = []
    for p in patient_dirs:
        pid = os.path.basename(p)
        try:
            t1    = glob.glob(os.path.join(p, "*t1.nii*"))[0]
            t1ce  = glob.glob(os.path.join(p, "*t1ce.nii*"))[0]
            t2    = glob.glob(os.path.join(p, "*t2.nii*"))[0]
            flair = glob.glob(os.path.join(p, "*flair.nii*"))[0]
            seg   = glob.glob(os.path.join(p, "*seg.nii*"))[0]
        except IndexError:
            continue
        datalist.append({"t1": t1, "t1ce": t1ce, "t2": t2, "flair": flair,
                          "label": seg, "id": f"brats_{pid}", "source": "glioma"})
    return datalist

brats_list = build_brats_datalist(BRATS_DIR)
print(f"BraTS usable cases in this chunk: {len(brats_list)}")

def build_ixi_datalist(folders):
    datalist = []
    for folder in folders:
        nii_files = glob.glob(os.path.join(folder, "*.nii")) + glob.glob(os.path.join(folder, "*.nii.gz"))
        if not nii_files:
            continue
        t2_path = nii_files[0]
        subj_id = os.path.basename(folder).replace(".nii", "")
        datalist.append({"t1": t2_path, "t1ce": t2_path, "t2": t2_path, "flair": t2_path,
                          "label": None, "id": f"ixi_{subj_id}", "source": "healthy"})
    return datalist

ixi_list = build_ixi_datalist(this_chunk_ixi_folders)
print(f"IXI usable cases in this chunk: {len(ixi_list)}")

full_datalist = brats_list + ixi_list
random.shuffle(full_datalist)
print(f"Total cases in chunk {CHUNK_ID}: {len(full_datalist)}")

assert len(brats_list) > 0, "No BraTS cases in this chunk."
assert len(ixi_list) > 0, "No IXI cases in this chunk."


# ============================================================
# BLOCK 6 — Preprocessing pipelines
# ============================================================
class RemapLabeld(MapTransform):
    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            d[key][d[key] == 4] = 3
        return d

preprocess_glioma = Compose([
    LoadImaged(keys=KEYS),
    EnsureChannelFirstd(keys=KEYS),
    Orientationd(keys=KEYS, axcodes="RAS", labels=(('L','R'),('P','A'),('I','S'))),
    Spacingd(keys=KEYS, pixdim=(1.5, 1.5, 1.5), mode=("bilinear",)*4 + ("nearest",)),
    CropForegroundd(keys=KEYS, source_key="t1"),
    Resized(keys=IMG_KEYS, spatial_size=TARGET_SIZE, mode="trilinear"),
    Resized(keys=["label"], spatial_size=TARGET_SIZE, mode="nearest"),
    RemapLabeld(keys=["label"]),
    NormalizeIntensityd(keys=IMG_KEYS, nonzero=True, channel_wise=True),
    ConcatItemsd(keys=IMG_KEYS, name="image"),
    ToTensord(keys=["image", "label"]),
])

preprocess_healthy = Compose([
    LoadImaged(keys=IMG_KEYS),
    EnsureChannelFirstd(keys=IMG_KEYS),
    Orientationd(keys=IMG_KEYS, axcodes="RAS", labels=(('L','R'),('P','A'),('I','S'))),
    Spacingd(keys=IMG_KEYS, pixdim=(1.5, 1.5, 1.5), mode="bilinear"),
    CropForegroundd(keys=IMG_KEYS, source_key="t1"),
    Resized(keys=IMG_KEYS, spatial_size=TARGET_SIZE, mode="trilinear"),
    NormalizeIntensityd(keys=IMG_KEYS, nonzero=True, channel_wise=True),
    ConcatItemsd(keys=IMG_KEYS, name="image"),
    ToTensord(keys=["image"]),
])


# ============================================================
# BLOCK 7 — Run preprocessing
# ============================================================
OUT_DIR = "/kaggle/working/processed"
shutil.rmtree(OUT_DIR, ignore_errors=True)
os.makedirs(OUT_DIR, exist_ok=True)

failed = []
start = time.time()

for i, item in enumerate(full_datalist):
    try:
        if item["source"] == "glioma":
            out = preprocess_glioma(item)
            image = out["image"].half()
            label = out["label"].to(torch.uint8).squeeze(0)
        else:
            out = preprocess_healthy(item)
            image = out["image"].half()
            label = torch.zeros(TARGET_SIZE, dtype=torch.uint8)

        torch.save({"image": image, "label": label, "id": item["id"], "source": item["source"]},
                   os.path.join(OUT_DIR, f"{item['id']}.pt"))

        if item["source"] == "glioma":
            shutil.rmtree(os.path.dirname(item["t1"]), ignore_errors=True)

        if i % 25 == 0:
            elapsed = (time.time() - start) / 60
            used_gb = shutil.disk_usage("/kaggle/working").used / 1e9
            free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
            print(f"[{i}/{len(full_datalist)}] saved {item['id']} ({item['source']}) - "
                  f"{elapsed:.1f} min elapsed - disk used: {used_gb:.1f}GB free: {free_gb:.1f}GB")
    except Exception as e:
        print(f"Failed on {item['id']}: {e}")
        failed.append(item["id"])

print(f"Done in {(time.time()-start)/60:.1f} min. {len(full_datalist)-len(failed)} saved, {len(failed)} failed.")

shutil.rmtree(BRATS_EXTRACT_DIR, ignore_errors=True)
!df -h /kaggle/working


# ============================================================
# BLOCK 8 — Sanity check
# ============================================================
files = os.listdir(OUT_DIR)
glioma_files = [f for f in files if f.startswith("brats")]
healthy_files = [f for f in files if f.startswith("ixi")]
print(f"Chunk {CHUNK_ID}: Glioma files saved: {len(glioma_files)} | Healthy files saved: {len(healthy_files)}")

if glioma_files:
    s = torch.load(os.path.join(OUT_DIR, glioma_files[0]), weights_only=False)
    print("Glioma sample:", s["image"].shape, s["image"].dtype, s["label"].dtype, s["label"].unique())
if healthy_files:
    s = torch.load(os.path.join(OUT_DIR, healthy_files[0]), weights_only=False)
    print("Healthy sample:", s["image"].shape, s["image"].dtype, s["label"].dtype, s["label"].unique())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 45.2 MB/s eta 0:00:00
Total BraTS patients available: 1251
Chunk 1: 626 BraTS patients (indices 625:1251)
Total IXI subjects available: 578
Chunk 1: 289 IXI subjects
Extracting chunk 1 patients from tar...
Extracting 3756 files for 626 patients...


/tmp/ipykernel_25/138238750.py:78: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=BRATS_EXTRACT_DIR, members=members_to_extract)


Extraction done in 0.6 min.
Resolved BRATS_DIR: /kaggle/working/brats2021_extracted
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  6.4G   14G  33% /kaggle/working
BraTS usable cases in this chunk: 626
IXI usable cases in this chunk: 289
Total cases in chunk 1: 915
[0/915] saved brats_BraTS2021_00095 (glioma) - 0.0 min elapsed - disk used: 6.8GB free: 14.1GB
[25/915] saved brats_BraTS2021_00305 (glioma) - 0.8 min elapsed - disk used: 6.8GB free: 14.1GB
[50/915] saved brats_BraTS2021_00240 (glioma) - 1.6 min elapsed - disk used: 6.8GB free: 14.2GB
[75/915] saved ixi_IXI517-IOP-1144-T2 (healthy) - 2.3 min elapsed - disk used: 6.8GB free: 14.1GB
[100/915] saved brats_BraTS2021_00099 (glioma) - 3.1 min elapsed - disk used: 6.8GB free: 14.1GB
[125/915] saved brats_BraTS2021_01627 (glioma) - 3.9 min elapsed - disk used: 6.8GB free: 14.1GB
[150/915] saved brats_BraTS2021_00781 (glioma) - 4.6 min elapsed - disk used: 6.9GB free: 14.0GB
[175/915] saved brats_BraTS2021_003